# Review workbench — dual-model → adjudicate → gold  (Phases 1–2)

Two models check each transcript, then **you** adjudicate and save the result as gold data:
**Model A** extracts (citation-validated), a different **Model B** reviews for missed/wrong
symptoms, you see the diff, accept/reject B's proposals, and save the signed-off extraction to
`gold/<id>.json` with full provenance.

Spec: `specs/review_and_gold.md`. Sections 2 & 3 each cost one API call (different model each).

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

from clinical import (
    get_transcript,
    DEFAULT_MODEL,
    REVIEWER_MODEL,
    REVIEW_SYSTEM_PROMPT,
    build_review_user_message,
    get_reviewer_model,
    show_review_diff,
    show_symptom_evidence_matrix,
    apply_accepted,
    build_gold_record,
    save_gold,
    gold_progress,
)
from graphs.validation_loop import build_graph

ENCOUNTER_DATE = "27.11.2025"
EXTRACTOR = build_graph()        # Model A: validation-loop extractor
REVIEWER = get_reviewer_model()  # Model B: independent reviewer

print(f"extractor (A): {DEFAULT_MODEL}")
print(f"reviewer  (B): {REVIEWER_MODEL}")
print(f"gold so far: {len(gold_progress())} transcript(s)")

## 1. Pick a transcript

In [ ]:
TID = "transcript_1"
transcript = get_transcript(TID)
print(transcript)

## 2. Model A extracts  ⚠️ API call

In [ ]:
result = EXTRACTOR.invoke({"transcript": transcript, "encounter_date": ENCOUNTER_DATE})
extraction = result["extraction"]
print(f"attempts={result['attempts']}  citation errors={len(result['errors'])}  "
      f"problems={len(extraction.problems)}")

## 3. Model B reviews A's extraction  ⚠️ API call (different model)

B reads the transcript + A's output and proposes what's **missing / wrong / misattributed** —
each backed by a verbatim excerpt.

In [ ]:
review_messages = [
    SystemMessage(content=REVIEW_SYSTEM_PROMPT),
    HumanMessage(content=build_review_user_message(
        transcript, extraction.model_dump_json(indent=2), ENCOUNTER_DATE)),
]
critique = REVIEWER.invoke(review_messages)
print(f"reviewer proposed {len(critique.changes)} change(s)")
print("overall:", critique.overall)

## 4. The diff — 🟩 captured by A · 🟧 reviewer says missed

In [ ]:
show_review_diff(transcript, extraction, critique)

## 5. The structured facts (A's extraction)

In [ ]:
show_symptom_evidence_matrix(extraction)

## 6. Adjudicate — accept / reject the reviewer's proposals

Read each numbered proposal below against the transcript and the diff above.

In [ ]:
for i, ch in enumerate(critique.changes):
    print(f"[{i}] {ch.kind:13} {ch.problem}  ·  field={ch.field}")
    print(f"      → {ch.text}")
    if ch.excerpt:
        print(f"      “{ch.excerpt}”")
    print(f"      reason: {ch.reason}")

Set `ACCEPT` to the indices you agree with, then merge. Accepted `missing` facts get added
to the right problem/field; `wrong`/`misattributed` are recorded but not auto-applied (yet).

In [ ]:
ACCEPT = []   # ← edit: e.g. [0, 2] to accept proposals 0 and 2

accepted = [critique.changes[i] for i in ACCEPT]
final_extraction, apply_notes = apply_accepted(extraction, accepted)
for n in apply_notes:
    print(("\u2713" if n["applied"] else "\u2022"), f"{n['field']}: {n['note']}")
print(f"\nfinal problems: {len(final_extraction.problems)}")

## 7. Review the merged result

In [ ]:
show_symptom_evidence_matrix(final_extraction)

## 8. Save as gold  ✅

Writes `gold/<transcript_id>.json` — the final extraction + both models + the critique + your
decisions + a timestamp. `gold/` is gitignored (it contains transcript text).

In [ ]:
decisions = [{"index": i, "accepted": (i in ACCEPT)} for i in range(len(critique.changes))]
record = build_gold_record(
    TID, final_extraction, critique, decisions,
    extractor_model=DEFAULT_MODEL, reviewer_model=REVIEWER_MODEL, encounter_date=ENCOUNTER_DATE,
)
path = save_gold(record)
print("saved", path)
print("gold dataset now:", gold_progress())

## Next — Phase 3

Wrap this as a LangGraph graph: extractor validation-loop as a subgraph, `interrupt()` for the
human step, and an apply-or-rebut cycle (the reviewer either incorporates a human-flagged fact or
argues why not). See `specs/review_and_gold.md`.